<a href="https://colab.research.google.com/github/DebStar17/GNSS-Anti-spoofing/blob/main/Final_GNSS_AI_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [71]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings('ignore')

# 1. CONFIGURATION
TRAIN_FILE = 'train.csv'
TEST_FILE = 'test.csv'
OUTPUT_FILE = 'output.csv'
intermediate_train = 'intermediate_train.csv'
intermediate_test = 'intermediate_test.csv'

# Features that focus on relative change, not absolute time
FEATURES = ['Physics_Score', 'c_pd', 'c_dd', 'c_constellation', 'c_te', 'c_cs']

In [72]:
# 2. DATA ENGINEERING
def load_and_engineer(file_path, is_train=True):
    print(f"--> Loading {file_path}...")
    df = pd.read_csv(file_path, low_memory=False)

    # 1. Basic Cleaning & Sorting
    df['Carrier_Doppler_hz'] = pd.to_numeric(df['Carrier_Doppler_hz'], errors='coerce')
    df = df.dropna(subset=['Carrier_Doppler_hz']).copy()

    numeric_cols = ['Carrier_Doppler_hz', 'Pseudorange_m', 'Carrier_phase', 'EC', 'LC', 'PC', 'PIP', 'PQP', 'CN0']
    for col in numeric_cols:
        df[col] = df[col].astype(float)

    df['time'] = df['time'].astype(int)
    if is_train:
        df['spoofed'] = df['spoofed'].astype(int)

    df = df.sort_values(by=['channel', 'time']).reset_index(drop=True)

    # --- RESET PROTECTION LOGIC ---
    # Identify if a row is part of a blackout or a fresh re-acquisition
    df['is_blackout'] = df['CN0'] == 0
    # A 'Reset' occurs if the current row or the previous row was a blackout
    df['is_reset'] = df.groupby('channel')['is_blackout'].transform(lambda x: x | x.shift(1).fillna(True))

    # 1. Calculate the change in meters and change in cycles independently
    df['p_delta'] = df.groupby('channel')['Pseudorange_m'].diff().abs()
    df['l_delta'] = df.groupby('channel')['Carrier_phase'].diff().abs()

    # Apply Reset Protection
    df.loc[df['is_reset'], ['p_delta', 'l_delta']] = 0
    df['velocity_mismatch'] = np.abs(df['p_delta'] - (df['l_delta'] * 0.19))

    df['pd_event'] = np.clip(df['velocity_mismatch'] / 10.0, 0, 1)
    df['c_pd'] = df.groupby('channel')['pd_event'].transform(lambda x: x.rolling(30, min_periods=1).max())


    # 2. Doppler_jump
    df['doppler_jump'] = df.groupby('channel')['Carrier_Doppler_hz'].diff().abs().fillna(0)
    df.loc[df['is_reset'], 'doppler_jump'] = 0
    # NEW: The jump 'event'
    df['dd_event'] = np.clip(df['doppler_jump'] / 50.0, 0, 1)
    # NEW: The model 'remembers' the jump for the next 30 seconds
    df['c_dd'] = df.groupby('channel')['dd_event'].transform(lambda x: x.rolling(30, min_periods=1).max())

    # 3. Constellation Logic
    time_group = df.groupby('time')['doppler_jump'].transform('mean')
    df['const_event'] = np.clip(time_group / 20.0, 0, 1)
    # The model remembers a global constellation takeover for 30 seconds
    df['c_constellation'] = df.groupby('channel')['const_event'].transform(lambda x: x.rolling(30, min_periods=1).max())


    # 4. Penalty/Probability Multiplier (p)
    df['p_signal'] = np.where(df['CN0'] > 25, 1.0, 0.5)
    df['p_signal'] = np.where(df['is_blackout'], 0.0, df['p_signal'])

    # 5. Tracking Error Stability (Rolling Mask)
    df['Tracking_Error'] = np.arctan2(df['PQP'], df['PIP'])

    # FIX: Assign valid_te to the dataframe so groupby can find it by name
    df['valid_te'] = df['Tracking_Error'].where(~df['is_blackout'])

    # Now reference the string name 'valid_te'
    df['te_stability'] = df.groupby('channel')['valid_te'].transform(
        lambda x: x.rolling(window=5, min_periods=3).std().fillna(0)
    )

    # Final cleanup: clip the stability score and drop the temp column
    df['c_te'] = np.clip(df['te_stability'] / 4.0, 0, 1)
    df = df.drop(columns=['valid_te'])


    # 6. Correlation Symmetry Bias Correction
    epsilon = 1e-9
    df['Corr_Symmetry'] = (df['EC'] - df['LC']) / (df['EC'] + df['LC'] + epsilon)
    df['cs_delta'] = df.groupby('channel')['Corr_Symmetry'].diff().abs().fillna(0)
    # Apply Reset Protection to symmetry
    df.loc[df['is_reset'], 'cs_delta'] = 0
    df['c_cs'] = np.clip(df['cs_delta'] / 1.0, 0, 1)


    # Weighted Score
    weights = {'pd': 0.5, 'dd': 0.5, 'const': 0.3, 'cs': 0.2, 'te': 0.2}
    df['Physics_Score'] = (
        df['c_pd'] * weights['pd'] +
        df['c_dd'] * weights['dd'] +
        df['c_constellation'] * weights['const'] +
        df['c_cs'] * weights['cs'] +
        df['c_te'] * weights['te']
    )*df['p_signal']

    df['Is_Signal_Dead'] = df['is_blackout'].astype(int)

    return df

In [73]:
# 3. PHASE 1: TRAINING
print("\n--- PHASE 1: TRAINING ---")
train_df = load_and_engineer(TRAIN_FILE, is_train=True)

train_df = train_df.sort_values(by='time')

# train_inter = train_df[['time', 'channel', 'spoofed'] + FEATURES].sort_values(by=['channel', 'time']).reset_index(drop=True)
# train_inter.to_csv(intermediate_train, index=False)
# print(f"Intermediate features saved to {intermediate_train}")

X_train = train_df[FEATURES]
y_train = train_df['spoofed']

print("--> Training Data loaded... Training Model")
rf_1 = RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1, class_weight='balanced')
train_filter = train_df['CN0'] > 0
rf_1.fit(X_train[train_filter], y_train[train_filter])

# --- IMPORTANCE LEADERBOARD ---
# 1. Extract importances
importances = rf_1.feature_importances_
feature_names = X_train.columns

# 2. Create a sorted DataFrame
feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

# 3. Print the Leaderboard
print("\nFeature Importance Leaderboard:")
print(feature_importance_df.to_string(index=False))


--- PHASE 1: TRAINING ---
--> Loading train.csv...
--> Training Data loaded... Training Model

Feature Importance Leaderboard:
        Feature  Importance
c_constellation    0.423965
           c_pd    0.238510
           c_dd    0.197322
  Physics_Score    0.073382
           c_te    0.037924
           c_cs    0.028897


In [74]:
print('Based on Training data')
# 4. Calculate Average Precision (Generalization Metric)
from sklearn.metrics import classification_report, average_precision_score
# Calculate Average Precision ONLY on the live signals to see true model performance
ap_score = average_precision_score(
    y_train[train_filter],
    rf_1.predict_proba(X_train[train_filter])[:, 1]
)
print(f"\nModel Average Precision (Live Signals): {ap_score:.4f}")

from sklearn.metrics import f1_score

# 2. Calculate the Weighted F1
weighted_f1 = f1_score(y_train[train_filter], rf_1.predict(X_train[train_filter]), average='weighted')

print(f"Model Weighted F1 Score: {weighted_f1:.4f}")

Based on Training data

Model Average Precision (Live Signals): 0.5632
Model Weighted F1 Score: 0.8082


In [75]:
print("\n--- PHASE 2: SCORING & STATE LATCHING WITH MEMORY ---")

# 1. Load and engineer test data
test_df = load_and_engineer(TEST_FILE, is_train=False)

# test_inter = test_df[['time', 'channel'] + FEATURES].sort_values(by=['channel', 'time']).reset_index(drop=True)
# test_inter.to_csv(intermediate_test, index=False)
# print(f"Intermediate features saved to {intermediate_test}")

# 2. Get RF probabilities (Using the features from your physics-aware model)
test_df['RF_Prob'] = rf_1.predict_proba(test_df[FEATURES])[:, 1]

# 3. Aggregating to Time-Level
# Max for RF_Prob (if one channel is spoofed, the constellation is at risk)
# Max for Is_Signal_Dead (if any channel is dead, we check for blackout)
result_agg = test_df.groupby('time').agg({
    'RF_Prob': 'max',
    'Is_Signal_Dead': 'mean'
}).reset_index().sort_values('time')

spoofed_final = []
confidence_scores = []
current_state = 0

# --- LATCH CONFIGURATION ---
LOOKBACK_WINDOW = 10  # Seconds to check for prior spoofing
DENSITY_THRESHOLD = 7 # Must be flagged at least 7 times
THRESHOLD = 0.65

for i, row in result_agg.iterrows():
    rf_prob = row['RF_Prob']
    is_signal_dead = row['Is_Signal_Dead']

    # Check the history of the last 10 seconds
    if i >= LOOKBACK_WINDOW:
        recent_history = spoofed_final[-LOOKBACK_WINDOW:]
        spoof_count = sum(recent_history)
        was_recently_spoofed = spoof_count >= DENSITY_THRESHOLD
    else:
        was_recently_spoofed = False

    # --- Determine the 'Spoofed' state ---
    if is_signal_dead < 1.0:
        # Normal operation: Use the RF probability
        if rf_prob > THRESHOLD:
            next_state = 1
        else:
            next_state = 0
    else:
        # BLACKOUT LOGIC: If we were being spoofed in the last 10s, maintain state
        if was_recently_spoofed:
            next_state = 1
        else:
            next_state = 0

    spoofed_final.append(next_state)

    # --- Confidence Calculation ---
    if is_signal_dead < 1.0:
        # Signal is live: Confidence based on RF probability
        confidence = rf_prob if next_state == 1 else (1 - rf_prob)
    else:
        confidence = 0.5

    confidence_scores.append(confidence)
    current_state = next_state

result_agg['Spoofed'] = spoofed_final
result_agg['Confidence'] = confidence_scores


--- PHASE 2: SCORING & STATE LATCHING WITH MEMORY ---
--> Loading test.csv...


In [76]:
# Final formatting
submission_final = result_agg[['time', 'Spoofed', 'Confidence']]
submission_final['time'] = submission_final['time'].astype(int)

submission_final.to_csv(OUTPUT_FILE, index=False)
print(f"\nDONE! '{OUTPUT_FILE}' is ready.")


DONE! 'output.csv' is ready.
